# 2 — Aplicar el modelo final al test

Carga lo que guardó el notebook 1 (`modelo/`), predice las 30.000 publicaciones de test y escribe `submission/submission.csv`.
Al final la compara con la submission enviada a Kaggle (`submission/submission_final.csv`).

En la banda, la fusión es TabPFN-3.5-Thinking por la API de Prior Labs: por defecto se usa su predicción guardada
(`data/thinking/test_banda.csv`, insumo fijo) y no se llama a la API (ver README).

**Requisitos:** GPU con CUDA y `TABPFN_TOKEN` (TabPFN predice leyendo su contexto).

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# Funciona corriendo desde la raíz del repositorio o desde notebooks/
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
CARPETA_MODELO = RAIZ / 'modelo'

from catboost import CatBoostClassifier

from banda_thinking import BANDA_THINKING_FIJA, BANDA_THINKING_NUEVA, guardar_matrices_banda, probabilidad_banda
from fusiones import ZONAS, ajustar_y_predecir, filas_de_test
from modelo_final import ajustar_y_predecir_tabpfn, predecir_catboost, predecir_lightgbm, predecir_meta

configuracion = json.loads((CARPETA_MODELO / 'configuracion.json').read_text())

# Banda Thinking: False = la predicción que generó la submission (insumo fijo). True = la corrida nueva por la API que deja
# src/banda_thinking.py en modelo/ (ver README; sube las filas de la banda a Prior Labs y consume créditos)
USAR_BANDA_THINKING_NUEVA = False

## 1. Capa 1: LightGBM, CatBoost y TabPFN combinados por el meta-modelo

In [ ]:
prediccion_lightgbm = predecir_lightgbm(joblib.load(CARPETA_MODELO / 'lightgbm.pkl'), pd.read_pickle(CARPETA_MODELO / 'test_lightgbm.pkl'))

modelo_catboost = CatBoostClassifier()
modelo_catboost.load_model(str(CARPETA_MODELO / 'catboost.cbm'))
prediccion_catboost = predecir_catboost(modelo_catboost, pd.read_pickle(CARPETA_MODELO / 'test_catboost.pkl'))

tabpfn = np.load(CARPETA_MODELO / 'tabpfn_contexto.npz')
inicio = time.time()
prediccion_tabpfn = ajustar_y_predecir_tabpfn(tabpfn['X'], tabpfn['y'], tabpfn['X_test'], configuracion['indices_categoricos_tabpfn'])
print(f'TabPFN: {(time.time() - inicio) / 60:.0f} min')

meta = joblib.load(CARPETA_MODELO / 'meta.pkl')
probabilidad_capa1 = predecir_meta(meta, [prediccion_lightgbm, prediccion_catboost, prediccion_tabpfn])
es_usado = probabilidad_capa1 > configuracion['umbral_resto']
print(f'capa 1: umbral {configuracion["umbral_resto"]:.3f} | usados {es_usado.mean():.4f}')

## 2. Capa 2: las fusiones reemplazan la predicción en las filas de banda, anillo y bolsillo

Anillo y bolsillo: TabPFN v3.5 con el contexto guardado. Banda: la predicción de TabPFN-3.5-Thinking; además se guardan sus
matrices de contexto y de test en `modelo/banda_thinking.npz`, que es lo que hay que subir a la API para volver a correrla.

In [ ]:
texto_grupo_test = np.load(CARPETA_MODELO / 'texto_grupo_test.npy')
for zona, configuracion_zona in ZONAS.items():
    guardado = np.load(CARPETA_MODELO / f'fusion_{zona}.npz')
    contexto = {'X': guardado['X'], 'y': guardado['y'], 'categoricos': guardado['categoricos'].tolist()}
    filas, X_zona = filas_de_test(zona, probabilidad_capa1, tabpfn['X_test'], configuracion['indices_categoricos_tabpfn'], texto_grupo_test)
    if configuracion_zona['modelo'] == 'tabpfn_thinking':
        guardar_matrices_banda(contexto, filas, X_zona)
        probabilidad = probabilidad_banda(filas, BANDA_THINKING_NUEVA if USAR_BANDA_THINKING_NUEVA else BANDA_THINKING_FIJA)
    else:
        probabilidad = ajustar_y_predecir(contexto, X_zona)
    es_usado[filas.fila.values] = probabilidad > configuracion_zona['corte']
    print(f'{zona}: {len(filas)} filas | usados {(probabilidad > configuracion_zona["corte"]).mean():.3f}')

## 3. Submission

In [ ]:
submission = pd.DataFrame({'ID': np.arange(1, len(es_usado) + 1), 'condition': np.where(es_usado, 'used', 'new')})
(RAIZ / 'submission').mkdir(exist_ok=True)
submission.to_csv(RAIZ / 'submission' / 'submission.csv', index=False)

enviada = pd.read_csv(RAIZ / 'submission' / 'submission_final.csv')
distintas = int((submission.condition != enviada.condition).sum())
print(f'submission/submission.csv: {len(submission)} filas | usados {es_usado.mean():.4f} | distintas de la enviada: {distintas}')